In [ ]:
#  single-hive KF baseline 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Paths

SPLIT_DIR = "../backend/data/splits_70_15_15"
TRAIN_PATH = f"{SPLIT_DIR}/train.parquet"
VAL_PATH   = f"{SPLIT_DIR}/val.parquet"
TEST_PATH  = f"{SPLIT_DIR}/test.parquet"

KF_DIR = Path("../backend/data/kf_outputs")
KF_DIR.mkdir(parents=True, exist_ok=True)
CFG_PATH = KF_DIR / "kf_run_config.csv"   
THR_PATH = KF_DIR / "kf_val_thresholds_demo_hive.csv"  


# Settings

OBS_COLS = ["temperature", "humidity", "audio_density"]
STATE_DIM = len(OBS_COLS)

DT_COL = "dt_prev_min"
BASE_DT_MIN = 15.0
USE_DT_AWARE_Q = True

MIN_USABLE_VAL  = 200
MIN_USABLE_TEST = 200

# Numerical stability
S_JITTER = 1e-6

# Load + CLEAN

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
    df["tag_number"] = pd.to_numeric(df["tag_number"], errors="coerce").astype("Int64")
    for c in OBS_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    if DT_COL in df.columns:
        df[DT_COL] = pd.to_numeric(df[DT_COL], errors="coerce")
    df = df.dropna(subset=["published_at", "tag_number"])
    df = df.sort_values(["tag_number", "published_at"]).reset_index(drop=True)
    return df

print("Loading splits...")
train_df = clean_df(pd.read_parquet(TRAIN_PATH))
val_df   = clean_df(pd.read_parquet(VAL_PATH))
test_df  = clean_df(pd.read_parquet(TEST_PATH))

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print("Hives:", int(train_df["tag_number"].nunique()))


# Helpers

def usable_rows(df_hive: pd.DataFrame) -> int:
    z = df_hive[OBS_COLS].to_numpy(float)
    return int(np.isfinite(z).any(axis=1).sum())

def last_observed_vector(z: np.ndarray) -> np.ndarray:
    """
    Per-dimension last observed value within this array (causal summary).
    Returns vector of length D with last finite values; NaN if never observed.
    """
    D = z.shape[1]
    last = np.full((D,), np.nan, dtype=float)
    for t in range(z.shape[0]):
        m = np.isfinite(z[t])
        last[m] = z[t, m]
    return last

def persistence_with_prevstate(z: np.ndarray, prev_last: np.ndarray | None) -> np.ndarray:
    
    T, D = z.shape
    out = np.full((T, D), np.nan, dtype=float)
    last = np.full((D,), np.nan, dtype=float) if prev_last is None else prev_last.copy()

    for t in range(T):
        out[t] = last
        m = np.isfinite(z[t])
        last[m] = z[t, m]
    return out

def rmse_per_dim(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    D = y_true.shape[1]
    out = np.full((D,), np.nan, dtype=float)
    for d in range(D):
        m = np.isfinite(y_true[:, d]) & np.isfinite(y_pred[:, d])
        if m.sum() > 0:
            out[d] = float(np.sqrt(np.mean((y_true[m, d] - y_pred[m, d]) ** 2)))
    return out

# z-RMSE aggregate (no unit mixing): normalise by TRAIN std per dimension
train_std = train_df[OBS_COLS].astype(float).std(skipna=True).to_numpy()
train_std = np.where(np.isfinite(train_std) & (train_std > 1e-12), train_std, 1.0)

def agg_zrmse(rmse_raw: np.ndarray) -> float:
    m = np.isfinite(rmse_raw) & np.isfinite(train_std) & (train_std > 1e-12)
    if m.sum() == 0:
        return np.nan
    return float(np.mean(rmse_raw[m] / train_std[m]))

def build_dt_scale(df_hive: pd.DataFrame):
    """
    dt scaling for Q (optional). Scales process noise when sampling interval increases.
    """
    if (not USE_DT_AWARE_Q) or (DT_COL not in df_hive.columns):
        return None
    dt = df_hive[DT_COL].to_numpy(float)
    s = dt / BASE_DT_MIN
    s = np.where(np.isfinite(s) & (s > 0), s, 1.0)
    s = np.clip(s, 1.0, 16.0)
    return s

def percentile_threshold(arr: np.ndarray, p: float = 95.0) -> float:
    """Compute percentile threshold ignoring NaNs; return np.nan if not enough values."""
    a = arr[np.isfinite(arr)]
    if a.size < 50:
        return np.nan
    return float(np.percentile(a, p))


train_hives = set(train_df["tag_number"].astype(int).unique())
val_hives   = set(val_df["tag_number"].astype(int).unique())
test_hives  = set(test_df["tag_number"].astype(int).unique())
common_hives = sorted(list(train_hives & val_hives & test_hives))

candidates = []
for h in common_hives:
    if usable_rows(val_df[val_df["tag_number"].astype(int) == h]) >= MIN_USABLE_VAL and \
       usable_rows(test_df[test_df["tag_number"].astype(int) == h]) >= MIN_USABLE_TEST:
        train_rows = int((train_df["tag_number"].astype(int) == h).sum())
        candidates.append((train_rows, int(h)))

if len(candidates) == 0:
    raise ValueError("No hive has enough usable VAL+TEST rows. Lower MIN_USABLE_* or inspect data.")

candidates.sort(reverse=True)
hive_id = candidates[0][1]
print("Demo hive chosen:", hive_id)

train_h = train_df[train_df["tag_number"].astype(int) == hive_id].reset_index(drop=True)
val_h   = val_df[val_df["tag_number"].astype(int) == hive_id].reset_index(drop=True)
test_h  = test_df[test_df["tag_number"].astype(int) == hive_id].reset_index(drop=True)
print("Hive splits:", train_h.shape, val_h.shape, test_h.shape)


A = np.eye(STATE_DIM)
H = np.eye(STATE_DIM)


z_tr = train_h[OBS_COLS].to_numpy(float)
dz = np.diff(z_tr, axis=0)
R_diag = 0.5 * np.nanvar(dz, axis=0)
R_diag = np.where(np.isfinite(R_diag) & (R_diag > 1e-8), R_diag, 1e-3)
R = np.diag(R_diag)
print("R diag:", np.diag(R))


# Load tuned Q_scale (preferred) else fallback

best_scale = None
if CFG_PATH.exists():
    cfg = pd.read_csv(CFG_PATH)
    if "best_Q_scale" in cfg.columns:
        best_scale = float(cfg.loc[0, "best_Q_scale"])
        print("Loaded best_Q_scale from tuning:", best_scale)

if best_scale is None or not np.isfinite(best_scale) or best_scale <= 0:
    best_scale = 0.1  # fallback
    print("Using fallback best_scale:", best_scale)

Q = np.diag(R_diag * best_scale)
print("Q diag:", np.diag(Q))


def init_from_first_valid(z: np.ndarray) -> np.ndarray:
    x0 = np.zeros((STATE_DIM,), dtype=float)
    for i in range(len(z)):
        m = np.isfinite(z[i])
        if m.any():
            x0[m] = z[i, m]
            break
    return x0

def seed_prev_last(prev_last: np.ndarray | None, z_current: np.ndarray) -> np.ndarray | None:
    
    if prev_last is None:
        return None
    out = prev_last.copy()
    if np.isfinite(out).all():
        return out
    x0 = init_from_first_valid(z_current)
    m = ~np.isfinite(out)
    out[m] = x0[m]
    return out

def kf_run_pred_filt(z: np.ndarray, Q_base: np.ndarray, R: np.ndarray, x0: np.ndarray, P0: np.ndarray, dt_scale=None):
    """
    Returns:
      x_pred[t], P_pred[t]  = forecast BEFORE seeing z[t] (FAIR one-step forecast)
      x_filt[t], P_filt[t]  = filtered AFTER assimilating z[t]
      nis[t]                = innovation-based score computed using forecast stats (FAIR)
      nis_dof[t]            = effective observed dimension at t
      nis_norm[t]           = nis / dof
    """
    T, D = z.shape
    I = np.eye(D)

    x_pred = np.zeros((T, D))
    P_pred = np.zeros((T, D, D))
    x_filt = np.zeros((T, D))
    P_filt = np.zeros((T, D, D))

    nis = np.full((T,), np.nan, dtype=float)
    nis_dof = np.zeros((T,), dtype=int)
    nis_norm = np.full((T,), np.nan, dtype=float)

    x = x0.copy()
    P = P0.copy()

    for t in range(T):
        # ---- predict ----
        x = A @ x

        scale = 1.0 if dt_scale is None else float(dt_scale[t])
        Q_t = Q_base * scale

        P = A @ P @ A.T + Q_t
        P = 0.5 * (P + P.T)

        x_pred[t] = x
        P_pred[t] = P

        # update 
        z_t = z[t]
        mask = np.isfinite(z_t)
        m_dim = int(mask.sum())
        nis_dof[t] = m_dim

        if m_dim > 0:
            H_t = H[mask, :]                    # (m, D)
            R_t = R[np.ix_(mask, mask)]         # (m, m)
            y = z_t[mask] - (H_t @ x)           # innovation

            S = H_t @ P @ H_t.T + R_t
            S = 0.5 * (S + S.T) + S_JITTER * np.eye(m_dim)

            # NIS (FAIR) using forecast stats
            try:
                nis[t] = float(y.T @ np.linalg.solve(S, y))
            except np.linalg.LinAlgError:
                nis[t] = float(y.T @ np.linalg.pinv(S) @ y)

            nis_norm[t] = nis[t] / max(m_dim, 1)

            # Kalman gain: solve(S, I_m) is stable
            try:
                K = (P @ H_t.T) @ np.linalg.solve(S, np.eye(m_dim))
            except np.linalg.LinAlgError:
                K = (P @ H_t.T) @ np.linalg.pinv(S)

            # state update
            x = x + K @ y

            # Joseph form covariance update
            P = (I - K @ H_t) @ P @ (I - K @ H_t).T + K @ R_t @ K.T
            P = 0.5 * (P + P.T)

        x_filt[t] = x
        P_filt[t] = P

    return x_pred, P_pred, x_filt, P_filt, nis, nis_dof, nis_norm


# Fair evaluation (NO thresholds)

P0 = np.eye(STATE_DIM) * 10.0

def eval_split(df_h, name, prev_last_obs=None):
    z = df_h[OBS_COLS].to_numpy(float)
    dt_scale = build_dt_scale(df_h)

    valid = int(np.isfinite(z).any(axis=1).sum())
    print(f"\n{name} coverage: rows={len(z)}, rows_with_any_obs={valid}")
    if (name == "VAL" and valid < MIN_USABLE_VAL) or (name == "TEST" and valid < MIN_USABLE_TEST):
        print(f"WARNING: {name} has low usable obs (valid={valid}). Results may be noisy.")

    # Baseline seeded from previous split end 
    prev_last_obs = seed_prev_last(prev_last_obs, z)
    base = persistence_with_prevstate(z, prev_last_obs)

    x0 = init_from_first_valid(z)

    x_pred, P_pred, x_filt, P_filt, nis, nis_dof, nis_norm = kf_run_pred_filt(
        z, Q, R, x0, P0, dt_scale=dt_scale
    )

    # FAIR forecast at time t is pre-update prediction:
    kf_fore = x_pred

    rmse_b = rmse_per_dim(z, base)
    rmse_k = rmse_per_dim(z, kf_fore)
    rmse_f = rmse_per_dim(z, x_filt)

    print(f"\n=== {name} (FAIR one-step forecast: pre-update at time t) ===")
    for i, c in enumerate(OBS_COLS):
        print(f"{c:>14} | baseline RMSE={rmse_b[i]:.4f} | KF RMSE={rmse_k[i]:.4f}")
    print(f"{'AGG z-RMSE':>14} | baseline={agg_zrmse(rmse_b):.4f} | KF={agg_zrmse(rmse_k):.4f}")

    print("\n(Filtered/assimilation fit — uses z[t], NOT a forecast)")
    for i, c in enumerate(OBS_COLS):
        print(f"{c:>14} | KF-filter RMSE={rmse_f[i]:.4f}")
    print(f"{'AGG z-RMSE':>14} | KF-filter={agg_zrmse(rmse_f):.4f}")

    last_obs = last_observed_vector(z)
    return z, base, kf_fore, P_pred, x_filt, nis, nis_dof, nis_norm, last_obs


# Run VAL + TEST with baseline continuity across splits

prev_last_val = last_observed_vector(train_h[OBS_COLS].to_numpy(float))

z_val, base_val, kf_val, Ppred_val, xf_val, nis_val, nis_dof_val, nis_norm_val, last_val_obs = eval_split(
    val_h, "VAL", prev_last_obs=prev_last_val
)

z_test, base_test, kf_test, Ppred_test, xf_test, nis_test, nis_dof_test, nis_norm_test, _ = eval_split(
    test_h, "TEST", prev_last_obs=last_val_obs
)


# Only use rows where dof>0 and NIS is finite
use_val_raw  = (nis_dof_val > 0) & np.isfinite(nis_val)
use_val_norm = (nis_dof_val > 0) & np.isfinite(nis_norm_val)

thr95_raw  = percentile_threshold(nis_val[use_val_raw], 95.0)
thr99_raw  = percentile_threshold(nis_val[use_val_raw], 99.0)

thr95_norm = percentile_threshold(nis_norm_val[use_val_norm], 95.0)
thr99_norm = percentile_threshold(nis_norm_val[use_val_norm], 99.0)

print("\n=== VAL-only anomaly thresholds (freeze these) ===")
print(f"NIS raw:   p95={thr95_raw:.4f}  p99={thr99_raw:.4f}")
print(f"NIS norm:  p95={thr95_norm:.4f} p99={thr99_norm:.4f}")

# Save thresholds 
pd.DataFrame([{
    "hive_id": int(hive_id),
    "best_Q_scale": float(best_scale),
    "R_diag_temp": float(R_diag[0]),
    "R_diag_humid": float(R_diag[1]),
    "R_diag_audio": float(R_diag[2]),
    "thr95_raw": float(thr95_raw) if np.isfinite(thr95_raw) else np.nan,
    "thr99_raw": float(thr99_raw) if np.isfinite(thr99_raw) else np.nan,
    "thr95_norm": float(thr95_norm) if np.isfinite(thr95_norm) else np.nan,
    "thr99_norm": float(thr99_norm) if np.isfinite(thr99_norm) else np.nan,
    "val_usable_raw": int(use_val_raw.sum()),
    "val_usable_norm": int(use_val_norm.sum()),
}]).to_csv(THR_PATH, index=False)
print("Saved thresholds ->", THR_PATH)

# Apply to TEST (DO NOT recompute thresholds on TEST)
is_anom_raw_test_p95  = (np.isfinite(nis_test) & (nis_test > thr95_raw)) if np.isfinite(thr95_raw) else np.zeros_like(nis_test, dtype=bool)
is_anom_norm_test_p95 = (np.isfinite(nis_norm_test) & (nis_norm_test > thr95_norm)) if np.isfinite(thr95_norm) else np.zeros_like(nis_norm_test, dtype=bool)

print("\n=== TEST anomaly flags using VAL thresholds (correct) ===")
if np.isfinite(thr95_raw):
    print(f"Using VAL raw NIS p95={thr95_raw:.4f} -> TEST flagged points: {int(is_anom_raw_test_p95.sum())}")
if np.isfinite(thr95_norm):
    print(f"Using VAL norm NIS p95={thr95_norm:.4f} -> TEST flagged points: {int(is_anom_norm_test_p95.sum())}")

# -----------------------------
# Plots (unchanged)
# -----------------------------
t_val = val_h["published_at"].to_numpy()

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for i, col in enumerate(OBS_COLS):
    axes[i].plot(t_val, z_val[:, i], label=f"Observed {col}")
    axes[i].plot(t_val, base_val[:, i], label="Baseline (missing-safe persistence)", alpha=0.7)
    axes[i].plot(t_val, kf_val[:, i], label="KF forecast (pre-update, FAIR)", linewidth=2)
    axes[i].set_ylabel(col)
    axes[i].legend()
axes[-1].set_xlabel("Time")
plt.suptitle(f"Hive {hive_id} — VAL: Forecast comparison (FAIR)")
plt.tight_layout()
plt.show()

i = 0  # temperature
plt.figure(figsize=(14, 4))
plt.plot(t_val, z_val[:, i], label="Observed", alpha=0.8)
plt.plot(t_val, base_val[:, i], label="Baseline (persistence)", alpha=0.8)
plt.plot(t_val, kf_val[:, i], label="KF forecast (FAIR)", linewidth=2)
plt.title(f"Hive {hive_id} — VAL: Baseline vs KF (Temperature)")
plt.xlabel("Time")
plt.ylabel("Temperature")
plt.legend()
plt.tight_layout()
plt.show()

N = 300
plt.figure(figsize=(14, 4))
plt.plot(t_val[-N:], z_val[-N:, i], label="Observed", alpha=0.8)
plt.plot(t_val[-N:], base_val[-N:, i], label="Baseline (persistence)", alpha=0.8)
plt.plot(t_val[-N:], kf_val[-N:, i], label="KF forecast (FAIR)", linewidth=2)
plt.title(f"Hive {hive_id} — VAL: Baseline vs KF (Temperature) [Zoom last {N} points]")
plt.xlabel("Time")
plt.ylabel("Temperature")
plt.legend()
plt.tight_layout()
plt.show()

std_pred_val = np.sqrt(np.clip(np.diagonal(Ppred_val, axis1=1, axis2=2), 1e-12, None))
dim = 0  # temperature
upper = kf_val[:, dim] + 2 * std_pred_val[:, dim]
lower = kf_val[:, dim] - 2 * std_pred_val[:, dim]

plt.figure(figsize=(14, 4))
plt.plot(t_val, kf_val[:, dim], label="KF forecast (temp)")
plt.plot(t_val, z_val[:, dim], label="Observed temp", alpha=0.7)
plt.fill_between(t_val, lower, upper, alpha=0.2, label="±2σ (forecast uncertainty)")
plt.title(f"Hive {hive_id} — KF forecast uncertainty (temperature)")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 3))
plt.plot(t_val, nis_val, label="KF NIS (raw, FAIR)")
if np.isfinite(thr95_raw):
    plt.axhline(thr95_raw, linestyle="--", label="VAL p95 (raw)")
plt.title(f"Hive {hive_id} — VAL NIS (raw)")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 3))
plt.plot(t_val, nis_norm_val, label="KF NIS / dof (normalized)")
if np.isfinite(thr95_norm):
    plt.axhline(thr95_norm, linestyle="--", label="VAL p95 (normalized)")
plt.title(f"Hive {hive_id} — VAL Normalized NIS")
plt.legend()
plt.tight_layout()
plt.show()

t_test = test_h["published_at"].to_numpy()

plt.figure(figsize=(14, 3))
plt.plot(t_test, nis_test, label="TEST NIS (raw)")
if np.isfinite(thr95_raw):
    plt.axhline(thr95_raw, linestyle="--", label="VAL p95 (raw) threshold")
plt.title(f"Hive {hive_id} — TEST NIS (raw) with VAL threshold")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 3))
plt.plot(t_test, nis_norm_test, label="TEST NIS / dof (normalized)")
if np.isfinite(thr95_norm):
    plt.axhline(thr95_norm, linestyle="--", label="VAL p95 (normalized) threshold")
plt.title(f"Hive {hive_id} — TEST Normalized NIS with VAL threshold")
plt.legend()
plt.tight_layout()
plt.show()


def make_export_df(df_h, z_true, z_pred, nis, nis_norm, dof, is_anom_raw=None, is_anom_norm=None):
    out = pd.DataFrame({
        "published_at": df_h["published_at"].to_numpy(),
        "tag_number": df_h["tag_number"].astype(int).to_numpy(),
    })
    for i, c in enumerate(OBS_COLS):
        out[f"{c}_true"] = z_true[:, i]
        out[f"{c}_pred_preupdate"] = z_pred[:, i]
    out["nis_raw"] = nis
    out["nis_norm"] = nis_norm
    out["nis_dof"] = dof
    if is_anom_raw is not None:
        out["is_anomaly_raw_p95"] = is_anom_raw.astype(int)
    if is_anom_norm is not None:
        out["is_anomaly_norm_p95"] = is_anom_norm.astype(int)
    return out

val_export = make_export_df(val_h, z_val, kf_val, nis_val, nis_norm_val, nis_dof_val)
test_export = make_export_df(
    test_h, z_test, kf_test, nis_test, nis_norm_test, nis_dof_test,
    is_anom_raw=is_anom_raw_test_p95,
    is_anom_norm=is_anom_norm_test_p95
)

val_path = KF_DIR / f"kf_val_hive_{hive_id}.parquet"
test_path = KF_DIR / f"kf_test_hive_{hive_id}.parquet"
val_export.to_parquet(val_path, index=False)
test_export.to_parquet(test_path, index=False)
print("\nExported:")
print(" -", val_path)
print(" -", test_path)


